In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set the font
from matplotlib import font_manager as fm
fpath = "Microsoft Aptos Fonts/Aptos.ttf"
prop = fm.FontProperties(fname=fpath)
#plt.rcParams['font.family'] = 'Aptos'

from matplotlib.ticker import FuncFormatter

First, we will check if the proportions even follow.

In [38]:
df['Value'].sum()

np.int64(6368770)

In [39]:
aspbi_data = {
    'A': 155221,
    'B': 36575,
    'C': 1202028,
    'D': 56455,
    'E': 43386,
    'F': 286849,
    'G': 1211342,
    'H': 221493,
    'I': 363970,
    'J': 169331,
    'K': 373866,
    'L': 104095,
    'M': 139732,
    'N': 1364700,
    'P': 298370,
    'Q': 215728,
    'R': 56864,
    'S': 68765
}

df = pd.DataFrame(list(aspbi_data.items()), columns=['Letter', 'Value'])
df.index = df['Letter']
values = df['Value']
values.sort_values(ascending=False).astype(int) / df['Value'].sum()

Letter
N    0.214280
G    0.190200
C    0.188738
K    0.058703
I    0.057149
P    0.046849
F    0.045040
H    0.034778
Q    0.033873
J    0.026588
A    0.024372
M    0.021940
L    0.016345
S    0.010797
R    0.008929
D    0.008864
E    0.006812
B    0.005743
Name: Value, dtype: float64

In [40]:
df['Value'].sum()

np.int64(6368770)

In [41]:
6_368_770

6368770

In [42]:
# Open and combine the January and April files
relevant_cols = ['PUFPWGTPRV', # Final Weight Based on Projection
                 'PUFC05_AGE', # Age
                 'PUFNEWEMPSTAT', # Employed (1), Unemployed (2), Not in the Labor Force (3)
                 'PUFC20_PWMORE', # Underemployed (1), NOT underemployed (2)
                 'PUFC23_PCLASS', # Self-Employed (3)
                 'PUFC07_GRADE', # Look at the code
                 'PUFC09_GRADTECH', # Went through TESDA (1), Did NOT go through TESDA (2)
                 'PUFC25_PBASIC', # Pay per Day
                 'PUFC31_FLWRK', # First time looking for work (1), no (2)
                 'PUFC14_PROCC',
                 'PUFC16_PKB',
]

filepath = 'data/PHL-PSA-LFS-2025-01-PUF/LFS PUF January 2025.CSV'
jan_df = pd.read_csv(filepath)

filepath = 'data/PHL-PSA-LFS-2025-04-PUF/LFS PUF April 2025.CSV'
apr_df = pd.read_csv(filepath)
#apr_df = apr_df[relevant_cols].replace(' ', np.nan)
#apr_df = apr_df[relevant_cols].replace('     ', np.nan)
#apr_df.dropna(inplace=True)

# combined_df = pd.concat([jan_df, apr_df])
# combined_df = combined_df[relevant_cols].replace(' ', np.nan)

In [44]:
lf_df['PUFC16_PKB'] = lf_df['PUFC16_PKB'].replace('  ', np.nan)

In [51]:
business_ranges = {
    "A": [(1, 3)],
    "B": [(5, 9)],
    "C": [(10, 33)],
    "D": [(35, 35)],
    "E": [(36, 39)],
    "F": [(41, 43)],
    "G": [(45, 47)],
    "H": [(49, 53)],
    "I": [(55, 56)],
    "J": [(58, 63)],
    "K": [(64, 66)],
    "L": [(68, 68)],
    "M": [(69, 75)],
    "N": [(77, 82)],
    "P": [(85, 85)],
    "Q": [(86, 88)],
    "R": [(90, 93)],
    "S": [(94, 96)],
    "Other": [(84, 84), (97, 98), (99, 99)]
}

def classify_business(code):
    if pd.isna(code):
        return "Other"

    code = int(code)
    for letter, ranges in business_ranges.items():
        for lower, upper in ranges:
            if lower <= code <= upper:
                return letter
    return "Other"

lf_df['Business_Letter'] = (
    lf_df['PUFC16_PKB']
    .map(classify_business)
)

In [64]:
is_private = lf_df['PUFC23_PCLASS'] == '1'
private_df = lf_df[is_private]

In [62]:
is_considered = ~(private_df['Business_Letter'] == 'Other')
private_df[is_considered]['PUFPWGTPRV'].sum()

np.float64(24013875.951100003)

In [63]:
private_df.groupby('Business_Letter')['PUFPWGTPRV'].sum().sort_values(ascending=False) / lf_df['PUFPWGTPRV'].sum()

Business_Letter
F        0.093323
G        0.067931
N        0.058678
A        0.056887
C        0.053166
H        0.035162
I        0.034711
K        0.012171
S        0.009860
Q        0.009133
P        0.008638
J        0.008523
R        0.007721
M        0.006905
L        0.003981
B        0.003332
D        0.002042
E        0.001152
Other    0.000008
Name: PUFPWGTPRV, dtype: float64